In [2]:
'''
POST endpoint: https://kartat.espoo.fi/teklaogcweb/WFS.ashx

BODY:
<?xml version="1.0" encoding="UTF-8"?>
<wfs:GetFeature
    service="WFS"
    version="1.0.0"
    outputFormat="GML3"
    xmlns:wfs="http://www.opengis.net/wfs"
    xmlns:GIS="http://www.tekla.com/schemas/GIS">

    <wfs:Query typeName="GIS:EcoCounter"/>
</wfs:GetFeature>
'''


import xml.etree.ElementTree as ET
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

xml_path = "data/espoo_eco_counter.xml"

# Namespaces (based on your XML structure)
ns = {
    "gml": "http://www.opengis.net/gml",
    "GIS": "http://www.tekla.com/schemas/GIS",
}

tree = ET.parse(xml_path)
root = tree.getroot()

rows = []
for eco in root.findall(".//GIS:EcoCounter", ns):
    name = eco.findtext("GIS:name", default="", namespaces=ns)

    # Bikes (PP)
    bike_prev_year = pd.to_numeric(
        eco.findtext("GIS:EDVUOSIPP", default=None, namespaces=ns),
        errors="coerce"
    )

    bike_curr_year = pd.to_numeric(
        eco.findtext("GIS:VUOSIPP", default=None, namespaces=ns),
        errors="coerce"
    )
    bike_curr_month = pd.to_numeric(
        eco.findtext("GIS:KKPP", default=None, namespaces=ns),
        errors="coerce"
    )
    bike_yesterday = pd.to_numeric(
        eco.findtext("GIS:EILINENPP", default=None, namespaces=ns),
        errors="coerce"
    )

    # Pedestrians (JK)
    ped_prev_year = pd.to_numeric(
        eco.findtext("GIS:EDVUOSIJK", default=None, namespaces=ns),
        errors="coerce"
    )

    ped_curr_year = pd.to_numeric(
        eco.findtext("GIS:VUOSIJK", default=None, namespaces=ns),
        errors="coerce"
    )
    ped_curr_month = pd.to_numeric(
        eco.findtext("GIS:KKJK", default=None, namespaces=ns),
        errors="coerce"
    )
    ped_yesterday = pd.to_numeric(
        eco.findtext("GIS:EILINENJK", default=None, namespaces=ns),
        errors="coerce"
    )

    pos = eco.find(".//gml:Point/gml:pos", ns)
    if pos is None:
        continue

    x, y = map(float, pos.text.split())

    rows.append({
        "name": name,
        "bike_prev_year": bike_prev_year,
        "bike_curr_year": bike_curr_year,
        "bike_curr_month": bike_curr_month,
        "bike_yesterday": bike_yesterday,
        "ped_prev_year": ped_prev_year,
        "ped_curr_year": ped_curr_year,
        "ped_curr_month": ped_curr_month,
        "ped_yesterday": ped_yesterday,
        "geometry": Point(x, y)
    })

# Create GeoDataFrame
gdf = gpd.GeoDataFrame(rows, geometry="geometry")

# Set CRS (GK25FIN) and convert to WGS84 for explore()
gdf = gdf.set_crs("EPSG:3879", allow_override=True).to_crs(4326)

# Drop rows without annual bike data
# gdf = gdf.dropna(subset=["bike_year"])
gdf.fillna(0, inplace=True)

# 🚲 Interactive map (bike counts only)
bike_m = gdf.explore(
    column="bike_prev_year",
    cmap="viridis",
    tooltip=["name", "bike_prev_year", "bike_curr_year", "bike_curr_month", "bike_yesterday"],
    tiles="CartoDB positron",
    marker_kwds={"radius": 6},
    legend=True,
)

bike_m

In [3]:
ped_m = gdf.explore(
    column="ped_prev_year",
    cmap="viridis",
    tooltip=["name", "ped_prev_year", "ped_curr_year", "ped_curr_month", "ped_yesterday"],
    tiles="CartoDB positron",
    marker_kwds={"radius": 6},
    legend=True,
)

ped_m